# Azure Read/Layout · Upstage Document OCR · PaddleOCR-VL 1.6 비교

기존 `handwriting_ocr_benchmark`의 **동일한 192개 S/O/A/P 필드 이미지**로 네 엔진을 비교합니다. NAVER CLOVA와 Mi:dm 후처리는 포함하지 않습니다.

- Azure: `prebuilt-read`, `prebuilt-layout`
- Upstage: Document OCR (`model=ocr`)
- 로컬 GPU: `PaddlePaddle/PaddleOCR-VL-1.6` (Transformers 직접 추론)
- API 키는 코드에 쓰지 않고 Colab Secrets에서만 읽습니다.
- 기본 자동 순위는 필드 크롭끼리만 계산합니다. 전체 페이지는 인쇄된 양식 글자까지 OCR되지만 현재 page reference는 손글씨 셀만 담고 있어, 옵션 실행 결과를 정성 검토용으로만 저장합니다.

> 합성 데이터 결과는 후보 압축용입니다. 최종 선택 전에는 개인정보를 제거한 실제 필기 표본으로 같은 평가를 반복하세요.

## 1. Drive 연결과 기존 프로젝트 찾기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_CANDIDATES = [
    Path('/content/drive/MyDrive/handwriting_ocr_benchmark_colab/handwriting_ocr_benchmark'),
    Path('/content/drive/MyDrive/handwriting_ocr_benchmark'),
    Path('/content/handwriting_ocr_benchmark_colab/handwriting_ocr_benchmark'),
    Path('/content/handwriting_ocr_benchmark'),
]
PROJECT_DIR = next((path for path in PROJECT_CANDIDATES if path.exists()), None)
assert PROJECT_DIR is not None, (
    'handwriting_ocr_benchmark 폴더를 찾지 못했습니다. '
    'PROJECT_CANDIDATES에 실제 Drive 경로를 추가하세요.'
)
assert (PROJECT_DIR / 'cloud_benchmark_adapters.py').exists(), (
    'cloud_benchmark_adapters.py가 없습니다. 노트북과 같은 배포본의 파일을 프로젝트 루트에 넣으세요.'
)
print('프로젝트:', PROJECT_DIR)
%cd $PROJECT_DIR

## 2. 의존성·데이터·GPU 확인

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -e . requests azure-ai-documentintelligence pandas
!python -m pip install -q --upgrade 'transformers>=5.0.0' 'Pillow>=10.0.0'

In [ ]:
import json
import torch

def jsonl_count(path):
    with Path(path).open(encoding='utf-8') as handle:
        return sum(1 for line in handle if line.strip())

counts = {
    'pages': jsonl_count('data/page_manifest.jsonl'),
    'fields': jsonl_count('data/field_manifest.jsonl'),
}
print('데이터:', counts)
assert counts == {'pages': 48, 'fields': 192}
assert torch.cuda.is_available(), 'PaddleOCR-VL 실행을 위해 Colab 런타임을 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

## 3. Colab Secrets와 실행 옵션

Colab 왼쪽 열쇠 아이콘에서 다음 이름을 추가하고 **Notebook access**를 켜세요.

- `AZURE_DOC_INTEL_ENDPOINT`
- `AZURE_DOC_INTEL_KEY`
- `UPSTAGE_API_KEY`

키가 없는 공급자는 자동으로 건너뜁니다. `RESET_RESULTS=True`는 아래 네 결과 JSONL만 다시 만들며 데이터셋은 지우지 않습니다.

In [ ]:
from google.colab import userdata

def read_secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

AZURE_ENDPOINT = read_secret('AZURE_DOC_INTEL_ENDPOINT')
AZURE_KEY = read_secret('AZURE_DOC_INTEL_KEY')
UPSTAGE_KEY = read_secret('UPSTAGE_API_KEY')

RUN_AZURE = bool(AZURE_ENDPOINT and AZURE_KEY)
RUN_UPSTAGE = bool(UPSTAGE_KEY)
RUN_PADDLE = True
RUN_PAGE_QUALITATIVE = False  # True면 전체 페이지 8개씩 추가 실행(자동 순위에는 미포함)
RESET_RESULTS = False
FIELD_LIMIT = None            # 빠른 시험: 8, 전체 평가: None
PAGE_LIMIT = 8

print({'azure': RUN_AZURE, 'upstage': RUN_UPSTAGE, 'paddle': RUN_PADDLE})
if not RUN_AZURE:
    print('주의: Azure Secrets가 없어 Azure Read/Layout을 건너뜁니다.')
if not RUN_UPSTAGE:
    print('주의: UPSTAGE_API_KEY가 없어 Upstage를 건너뜁니다.')

## 4. 공통 러너

각 엔진은 첫 표본을 먼저 시험하고, 성공하면 같은 결과 파일에 이어서 씁니다. 중간에 런타임이 끊겨도 `resume=True`로 완료된 표본을 재사용하므로 API 중복 호출을 줄입니다.

In [ ]:
import sys
from collections import Counter

sys.path.insert(0, str(PROJECT_DIR))
from cloud_benchmark_adapters import (
    AzureDocumentIntelligenceAdapter,
    UpstageDocumentOCRAdapter,
)
from soapbench.dataset import read_jsonl
from soapbench.runner import run_inference

FIELD_MANIFEST = Path('data/field_manifest.jsonl')
PAGE_MANIFEST = Path('data/page_manifest.jsonl')
RUN_DIR = Path('runs/azure-upstage-paddlevl')
RUN_DIR.mkdir(parents=True, exist_ok=True)

FIELD_OUTPUTS = {
    'azure-read': RUN_DIR / 'azure-read-field.jsonl',
    'azure-layout': RUN_DIR / 'azure-layout-field.jsonl',
    'upstage': RUN_DIR / 'upstage-document-ocr-field.jsonl',
    'paddle': RUN_DIR / 'paddleocr-vl16-field.jsonl',
}
PAGE_OUTPUTS = {}

if RESET_RESULTS:
    for path in FIELD_OUTPUTS.values():
        path.unlink(missing_ok=True)

def rows_from(path):
    return list(read_jsonl(path)) if Path(path).exists() else []

def run_verified(adapter, manifest, output, limit=None):
    manifest = Path(manifest)
    output = Path(output)
    expected = min(jsonl_count(manifest), limit) if limit is not None else jsonl_count(manifest)
    previous = rows_from(output)
    if previous:
        stale = [row for row in previous if row.get('model') != adapter.name]
        if stale:
            raise RuntimeError(f'{output}에 다른 모델 결과가 있습니다. RESET_RESULTS=True로 다시 실행하세요.')
        old_errors = [row for row in previous if row.get('error')]
        if old_errors:
            raise RuntimeError(f'{output}에 이전 오류가 있습니다. RESET_RESULTS=True로 다시 실행하세요: {old_errors[0]["error"]}')
    else:
        smoke = run_inference(manifest=manifest, adapter=adapter, output=output, limit=1, resume=False)
        first = rows_from(output)[0]
        if smoke['failed'] or first.get('error'):
            raise RuntimeError(f'{adapter.name} 1개 시험 실패: {first.get("error")}')
        print(f'{adapter.name}: 1개 시험 성공')

    result = run_inference(
        manifest=manifest, adapter=adapter, output=output, limit=limit, resume=True
    )
    rows = rows_from(output)
    errors = [row for row in rows if row.get('error')]
    assert len(rows) == expected, f'{output}: {len(rows)}/{expected}개만 저장됨'
    if errors:
        counts = Counter(row['error'] for row in errors)
        raise RuntimeError(f'{adapter.name} 오류 {len(errors)}개: {counts.most_common(3)}')
    print(adapter.name, result, '총 표본=', len(rows))
    return output

## 5. Azure Read — 동일 필드 192개

In [ ]:
azure_read = None
if RUN_AZURE:
    azure_read = AzureDocumentIntelligenceAdapter(
        'prebuilt-read', endpoint=AZURE_ENDPOINT, key=AZURE_KEY
    )
    run_verified(azure_read, FIELD_MANIFEST, FIELD_OUTPUTS['azure-read'], FIELD_LIMIT)

## 6. Azure Layout — 동일 필드 192개

필드 크롭에서는 Layout의 구조 분석 이점이 거의 없으므로 Read와 결과가 비슷할 수 있습니다. 그래도 입력·평가지표를 동일하게 맞춰 인식 성능과 지연시간을 직접 비교합니다.

In [ ]:
azure_layout = None
if RUN_AZURE:
    azure_layout = AzureDocumentIntelligenceAdapter(
        'prebuilt-layout', endpoint=AZURE_ENDPOINT, key=AZURE_KEY
    )
    run_verified(azure_layout, FIELD_MANIFEST, FIELD_OUTPUTS['azure-layout'], FIELD_LIMIT)

## 7. Upstage Document OCR — 동일 필드 192개

현재 공개 단가가 페이지당 `$0.0015`라면 192개 전체 실행은 약 `$0.288`(세금 제외)입니다. 실제 결제 전 콘솔의 최신 단가와 무료 크레딧을 확인하세요.

In [ ]:
upstage = None
if RUN_UPSTAGE:
    upstage = UpstageDocumentOCRAdapter(api_key=UPSTAGE_KEY)
    run_verified(upstage, FIELD_MANIFEST, FIELD_OUTPUTS['upstage'], FIELD_LIMIT)

## 8. PaddleOCR-VL 1.6 — 동일 필드 192개

모델을 한 번만 메모리에 올린 뒤 1개 시험과 전체 실행을 이어서 수행합니다.

In [ ]:
paddle = None
if RUN_PADDLE:
    from soapbench.adapters import PaddleOCRVLAdapter
    paddle = PaddleOCRVLAdapter(
        device='cuda', engine='transformers', pipeline_version='v1.6'
    )
    run_verified(paddle, FIELD_MANIFEST, FIELD_OUTPUTS['paddle'], FIELD_LIMIT)

## 9. 필드 기준 통합 리포트

CER는 낮을수록, critical accuracy는 높을수록 좋습니다. API 모델의 latency에는 인터넷 왕복과 Azure polling 시간이 포함되므로 실제 배포 위치에서 한 번 더 재세요.

In [ ]:
import subprocess
import pandas as pd
from IPython.display import HTML, display

prediction_files = [path for path in FIELD_OUTPUTS.values() if path.exists()]
assert len(prediction_files) >= 2, '비교할 성공 결과가 2개 미만입니다. 위 실행 셀을 확인하세요.'
for path in prediction_files:
    rows = rows_from(path)
    assert rows and not any(row.get('error') for row in rows), f'{path}에 오류가 있습니다.'

REPORT_DIR = Path('reports/azure-upstage-paddlevl/field')
cmd = [sys.executable, '-m', 'soapbench', 'evaluate', '--predictions']
cmd += [str(path) for path in prediction_files]
cmd += ['--output-dir', str(REPORT_DIR)]
subprocess.run(cmd, check=True)

summary = pd.read_csv(REPORT_DIR / 'summary.csv')
display(summary.sort_values(['cer', 'critical_accuracy'], ascending=[True, False]))
display(HTML(filename=str(REPORT_DIR / 'report.html')))

## 10. 선택: 전체 페이지 정성 검토

`RUN_PAGE_QUALITATIVE=True`일 때 Layout, Upstage, Paddle을 페이지에 실행해 JSONL만 저장합니다. 현재 페이지 정답은 손글씨 셀만 포함하지만 OCR 결과에는 인쇄된 양식 문구도 들어가므로, 이 결과에 CER 순위를 매기면 잘못된 결론이 납니다.

In [ ]:
if RUN_PAGE_QUALITATIVE:
    page_jobs = []
    if azure_layout is not None:
        page_jobs.append((azure_layout, RUN_DIR / 'azure-layout-page-qualitative.jsonl'))
    if upstage is not None:
        page_jobs.append((upstage, RUN_DIR / 'upstage-page-qualitative.jsonl'))
    if paddle is not None:
        page_jobs.append((paddle, RUN_DIR / 'paddleocr-vl16-page-qualitative.jsonl'))
    for adapter, output in page_jobs:
        if RESET_RESULTS:
            output.unlink(missing_ok=True)
        PAGE_OUTPUTS[adapter.name] = run_verified(
            adapter, PAGE_MANIFEST, output, PAGE_LIMIT
        )
    for model_name, path in PAGE_OUTPUTS.items():
        sample = rows_from(path)[0]
        print('\n---', model_name, '/', sample['sample_id'], '---')
        print(sample['text'][:1500])
else:
    print('전체 페이지 정성 검토를 건너뜁니다.')

## 11. 리포트와 원본 예측 JSONL 다운로드

ZIP에는 API 키나 원본 공급자 응답이 아니라, 벤치마크 텍스트 결과·지연시간·오류 필드와 HTML/CSV 리포트만 들어갑니다.

In [ ]:
import shutil
import tempfile
from google.colab import files

bundle = Path(tempfile.mkdtemp(prefix='ocr_bakeoff_'))
shutil.copytree(REPORT_DIR, bundle / 'report')
(bundle / 'runs').mkdir()
for path in prediction_files + list(PAGE_OUTPUTS.values()):
    shutil.copy2(path, bundle / 'runs' / path.name)
archive = shutil.make_archive('/content/azure_upstage_paddlevl_results', 'zip', bundle)
print('다운로드:', archive)
files.download(archive)